In [1]:
# step 1 ) update config.yaml

In [2]:
# step 2 ) update param.yaml

In [3]:
import os 

In [4]:
os.getcwd()

'e:\\projects\\DL_kidney_tumor_pred\\kidney_tumor_prediction_project\\research'

In [6]:
os.chdir("../")

In [7]:
os.getcwd()

'e:\\projects\\DL_kidney_tumor_pred\\kidney_tumor_prediction_project'

In [8]:
# step 3 ) update entity

In [9]:
from dataclasses import dataclass
from pathlib import Path




@dataclass(frozen=True)
class DataIngestionConfig:
   root_dir: Path
   source_URL: str
   local_data_file: Path
   unzip_dir: Path


In [10]:
# step 4) update configuration manager 

In [11]:
from src.cnnClassifier.constants import * 
from src.cnnClassifier.utils.common import read_yaml , create_directories

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = Path("config/config.yaml"),
        params_filepath = Path("params.yaml")):


        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)


        create_directories([self.config.artifacts_root])




   
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion


        create_directories([config.root_dir])


        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )


        return data_ingestion_config




In [13]:
# step 5 ) components 

In [14]:
import os 
import zipfile
import gdown
from src.cnnClassifier import logger 
from src.cnnClassifier.utils.common import get_size

In [15]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


   
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''


        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")


            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)


            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")


        except Exception as e:
            raise e
       
   


    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)




In [16]:
# step 6 ) Pipeline 

In [17]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-03-24 18:14:46,157: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-24 18:14:46,161: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 18:14:46,163: INFO: common: created directory at: artifacts]
[2026-03-24 18:14:46,165: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-24 18:14:46,166: INFO: 1499037397: Downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=99a10661-0b92-4a44-8129-5d9c7aa8c285
To: e:\projects\DL_kidney_tumor_pred\kidney_tumor_prediction_project\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:06<00:00, 9.51MB/s]

[2026-03-24 18:14:55,496: INFO: 1499037397: Downloaded data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]
